# Extended source sensitivity calculator

This notebook show how to compute a extended source sensitivity for a specific model. In the current status of cosipy, we are limited by the available responses, i.e at 511, 1157, 1173, 1333 and 1809 keV and a coarse continuum. Keep in mind that due to the coarseness of the current responses, the sensitivity might be underestimated. 

The collaboration is currently developping a neural network approach for the response to use with an unbinned analysis. This will allow us to fully exploit COSI power. 

In [2]:
import os
import logging
import sys

logger = logging.getLogger('cosipy')
logger.setLevel(logging.INFO)
logger.addHandler(logging.StreamHandler(sys.stdout))

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"


from cosipy.spacecraftfile import SpacecraftHistory
from cosipy.response.FullDetectorResponse import FullDetectorResponse
from cosipy.response import PointSourceResponse,ExtendedSourceResponse
from cosipy.util import fetch_wasabi_file

from cosipy.statistics import PoissonLikelihood
from cosipy.background_estimation import FreeNormBinnedBackground
from cosipy.interfaces import ThreeMLPluginInterface
from cosipy.response import BinnedThreeMLModelFolding, BinnedInstrumentResponse, BinnedThreeMLExtendedSourceResponse
from cosipy.data_io import EmCDSBinnedData
from cosipy.source_injector import SourceInjector
from astromodels import SpatialTemplate_2D_Healpix
from histpy import Histogram
from mhealpy import HealpixMap

from scoords import SpacecraftFrame

from astropy.time import Time
import astropy.units as u
from astropy.coordinates import SkyCoord, Galactic

import numpy as np
import matplotlib.pyplot as plt

from threeML import *
from astromodels import *

from pathlib import Path



21:57:11 WARNING   The naima package is not available. Models that depend on it will not be         ]8;id=191395;file:///uni-mainz.de/homes/sgallego/software/COSItools/cosipy_interfaces/Pyenv/astromodels/astromodels/functions/functions_1D/functions.py\functions.py]8;;\:]8;id=328868;file:///uni-mainz.de/homes/sgallego/software/COSItools/cosipy_interfaces/Pyenv/astromodels/astromodels/functions/functions_1D/functions.py#43\43]8;;\
                  available                                                                                        

         WARNING   The GSL library or the pygsl wrapper cannot be loaded. Models that depend on it  ]8;id=956489;file:///uni-mainz.de/homes/sgallego/software/COSItools/cosipy_interfaces/Pyenv/astromodels/astromodels/functions/functions_1D/functions.py\functions.py]8;;\:]8;id=577703;file:///uni-mainz.de/homes/sgallego/software/COSItools/cosipy_interfaces/Pyenv/astromodels/astromodels/functions/functions_1D/functions.py#65\65]8;;\
                  will not be available.                                                                           

         WARNING   The ebltable package is not available. Models that depend on it will not be     ]8;id=207928;file:///uni-mainz.de/homes/sgallego/software/COSItools/cosipy_interfaces/Pyenv/astromodels/astromodels/functions/functions_1D/absorption.py\absorption.py]8;;\:]8;id=887405;file:///uni-mainz.de/homes/sgallego/software/COSItools/cosipy_interfaces/Pyenv/astromodels/astromodels/functions/functions_1D/absorption.py#33\33]8;;\
                  available                                                                                        

21:57:12 INFO      Starting 3ML!                                                                     ]8;id=445621;file:///uni-mainz.de/homes/sgallego/software/COSItools/cosipy_interfaces/Pyenv/lib/python3.11/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=2376;file:///uni-mainz.de/homes/sgallego/software/COSItools/cosipy_interfaces/Pyenv/lib/python3.11/site-packages/threeML/__init__.py#44\44]8;;\

         WARNING   WARNINGs here are NOT errors                                                      ]8;id=884647;file:///uni-mainz.de/homes/sgallego/software/COSItools/cosipy_interfaces/Pyenv/lib/python3.11/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=487671;file:///uni-mainz.de/homes/sgallego/software/COSItools/cosipy_interfaces/Pyenv/lib/python3.11/site-packages/threeML/__init__.py#45\45]8;;\

...

         WARNING   PyGMO is not available                                                      ]8;id=339497;file:///uni-mainz.de/homes/sgallego/software/COSItools/cosipy_interfaces/Pyenv/lib/python3.11/site-packages/threeML/minimizer/minimization.py\minimization.py]8;;\:]8;id=547026;file:///uni-mainz.de/homes/sgallego/software/COSItools/cosipy_interfaces/Pyenv/lib/python3.11/site-packages/threeML/minimizer/minimization.py#1228\1228]8;;\

21:57:13 WARNING   The cthreeML package is not installed. You will not be able to use plugins which  ]8;id=960280;file:///uni-mainz.de/homes/sgallego/software/COSItools/cosipy_interfaces/Pyenv/lib/python3.11/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=904170;file:///uni-mainz.de/homes/sgallego/software/COSItools/cosipy_interfaces/Pyenv/lib/python3.11/site-packages/threeML/__init__.py#95\95]8;;\
                  require the C/C++ interface (currently HAWC)                                                     

         WARNING   Could not import plugin FermiLATLike.py. Do you have the relative instrument     ]8;id=810873;file:///uni-mainz.de/homes/sgallego/software/COSItools/cosipy_interfaces/Pyenv/lib/python3.11/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=810971;file:///uni-mainz.de/homes/sgallego/software/COSItools/cosipy_interfaces/Pyenv/lib/python3.11/site-packages/threeML/__init__.py#136\136]8;;\
                  software installed and configured?                                                               

         WARNING   Could not import plugin HAWCLike.py. Do you have the relative instrument         ]8;id=899814;file:///uni-mainz.de/homes/sgallego/software/COSItools/cosipy_interfaces/Pyenv/lib/python3.11/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=714079;file:///uni-mainz.de/homes/sgallego/software/COSItools/cosipy_interfaces/Pyenv/lib/python3.11/site-packages/threeML/__init__.py#136\136]8;;\
                  software installed and configured?                                                               

21:57:14 WARNING   No fermitools installed                                              ]8;id=768676;file:///uni-mainz.de/homes/sgallego/software/COSItools/cosipy_interfaces/Pyenv/lib/python3.11/site-packages/threeML/utils/data_builders/fermi/lat_transient_builder.py\lat_transient_builder.py]8;;\:]8;id=556661;file:///uni-mainz.de/homes/sgallego/software/COSItools/cosipy_interfaces/Pyenv/lib/python3.11/site-packages/threeML/utils/data_builders/fermi/lat_transient_builder.py#44\44]8;;\

# Download the data

In [2]:
data_path = Path("") # /path/to/files. Current dir by default

In [ ]:
#Download the Orientation file
fetch_wasabi_file('COSI-SMEX/develop/Data/Orientation/DC3_final_530km_3_month_with_slew_15sbins_GalacticEarth_SAA.fits', checksum = 'e86df2407eb052cf0c1db4a8e7598727')

In [3]:
#Download the extended sourde response
#Choose the response depending on your model

#Model for a 511 keV line
#fetch_wasabi_file('COSI-SMEX/DC3/Data/Responses/extended_source_response/extended_source_response_511_merged.h5.gz', unzip=True,checksum = 'afcf551d3a8f800a268bef1490d0d48a')

#Model for a 1157 keV line
#fetch_wasabi_file('COSI-SMEX/DC3/Data/Responses/extended_source_response/extended_source_response_Ti44_merged.h5.gz',unzip=True,checksum = '7ac6e1d9651ae33cccf94102f5356d9e')

#Model for a 1173 keV line
#fetch_wasabi_file('COSI-SMEX/DC3/Data/Responses/extended_source_response/extended_source_response_Fe60_low_merged.h5.gz', unzip=True ,checksum = '2562f2ac0be44a60ea2d1adcea8e3b4a')

#Model for a 1333 keV line
#fetch_wasabi_file('COSI-SMEX/DC3/Data/Responses/extended_source_response/extended_source_response_Fe60_high_merged.h5.gz', unzip=True ,checksum = '037217db2e8ca8a9a74a247e8fab9eb3')

#Model for a 1809 keV line
fetch_wasabi_file('COSI-SMEX/DC3/Data/Responses/extended_source_response/extended_source_response_Al26_merged.h5.gz',unzip=True,checksum = '218d088159c6e11d0ddecc03ca9fa399')


#Model for an other line or continuum source
#fetch_wasabi_file('COSI-SMEX/DC3/Data/Responses/extended_source_response/extended_source_response_continuum_merged.h5.gz' ,unzip=True,checksum = 'a3a4a57acfbfe00378507441fa7a5891')



ClientError: An error occurred (404) when calling the HeadObject operation: Not Found

In [ ]:
#Download the background model 
#Choose the file depending on your model

#Model for a 511 keV line
#fetch_wasabi_file('COSI-SMEX/DC4/Data/Backgrounds/gal_DC4_bkg_511_binned_smoothed_fwhm_10.0deg.hdf5', checksum = '1b437be9b4d9e85a230dfe4f4d2ca67e')

#Model for a 1157 keV line
#fetch_wasabi_file('COSI-SMEX/DC4/Data/Backgrounds/gal_DC4_bkg_Ti44_binned_smoothed_fwhm_10.0deg.hdf5',checksum = 'bd62bfde46628a6e68434a6c68c7167d')

#Model for a 1173 keV line
#fetch_wasabi_file('COSI-SMEX/DC4/Data/Backgrounds/gal_DC4_bkg_Fe60_low_binned_smoothed_fwhm_10.0deg.hdf5',checksum = '57f972b0a1c2ece4626c85ced3cc050f')

#Model for a 1333 keV line
#fetch_wasabi_file('COSI-SMEX/DC4/Data/Backgrounds/gal_DC4_bkg_Fe60_high_binned_smoothed_fwhm_10.0deg.hdf5',checksum = 'db56a9fd9d8fc37cb03fba5e907a80c3')

#Model for a 1809 keV line
fetch_wasabi_file('COSI-SMEX/DC4/Data/Backgrounds/gal_DC4_bkg_Al26_binned_smoothed_fwhm_10.0deg.hdf5',checksum = 'bd6a32f8abee5b989b87149111487faa')


#Model for an other line or continuum source
#fetch_wasabi_file('COSI-SMEX/DC4/Data/Backgrounds/gal_DC4_bkg_continuum_binned_smoothed_fwhm_10.0deg.hdf5',checksum = 'b06599be5e32a53a6afd96a13e9ff37a')




# Define your source model

For an extended source we need to define a spectrum shape and a spatial shape : 

### Spectrum Shape
For this example we are using a Gaussian but other functions can be used (see https://astromodels.readthedocs.io/en/latest/notebooks/function_list.html#Included-1D-Functions) 

Additionaly you can also provide your own spectrum template from a file. See Template (Table) Models in :
https://threeml.readthedocs.io/en/stable/notebooks/spectral_models.html

Threeml expect the spectrum shape to have units of cm2/s/keV

In [29]:
F = 3e-5 / u.cm / u.cm / u.s  
mu = 1809*u.keV
sigma = 0.3*u.keV
spectrum = Gaussian()
spectrum.F.value = F.value
spectrum.F.unit = F.unit
spectrum.mu.value = mu.value
spectrum.mu.unit = mu.unit
spectrum.sigma.value = sigma.value
spectrum.sigma.unit = sigma.unit

# Set spectral parameters for fitting:
spectrum.F.free = True
spectrum.mu.free = False
spectrum.sigma.free = False

#set limit for the param
spectrum.F.min_value = 0
spectrum.F.max_value = 1



### Spatial shape

For the spatial shape you need to provide a Healpix map in a fits file. The map should be normalized to the pixel area so that sum(map)*pixelarea = 1 /sr

In [19]:
#For this example we will create here the spatial shape model but the fits file containing the spatial distribution can come
#from any source (ex : CLUMPY) 
skymap = HealpixMap(nside = 16, scheme = "ring", dtype = float, coordsys='G')
skymap[:] = 1 #fill the map with 1, so no particular spatial distribution


#normalized the map to 1/sr
pix_area = skymap.pixarea().value
skymap[:] = skymap[:]/(np.sum(skymap)*pix_area)


#save the map example to a fits file
skymap.write_map("HealpixMap_example.fits",overwrite=True)

#declare the spatial shape
spatial_shape =  SpatialTemplate_2D_Healpix(fits_file="HealpixMap_example.fits")


## Extended source model
Now that we declare our spectral shape and spatial shape, we can create our extended source model

In [31]:
source = ExtendedSource('source', spectral_shape=spectrum, spatial_shape=spatial_shape)


model = Model(source)  # Model with single source. If we had multiple sources, we would do Model(source1, source2, ...)


# Open the Response, Orientation and background files

In [27]:
#open the Response file

#replace the background and response file by the one you want to use
extendedsourceresponse = "extended_source_response_Al26_merged.h5"
bkgfile = "gal_DC4_bkg_Al26_binned_smoothed_fwhm_10.0deg.hdf5"
sc_orientation_path = "DC3_final_530km_3_month_with_slew_15sbins_GalacticEarth_SAA.fits"

#open the extended source response
dr = ExtendedSourceResponse.open(data_path / extendedsourceresponse )

#Create the extended source injector 
injector = SourceInjector(response_path=data_path / extendedsourceresponse, response_frame = "galactic")

#open the background model 
bg_model = Histogram.open(data_path / bkgfile)

#open the Orientation file
sc_orientation = SpacecraftHistory.open(data_path / sc_orientation_path)

# Compute the sensitivity

The Likelihood (LH) fit is ran hundred times (more would be better but depending on how fast it is you could increase this number).
Each time a new signal dataset is created by convolving the model with the response and applying some poisson statistic (also called the source injector). Similarly for the background, a dataset is created by taking a poisson sample of the background model.

For each iteration the LH ratio test is computed : 

$$
\text{LHtest} = -2 * (L_0 - L_1)
$$

$L_0$ being the LH value for the background only hypothesis and $L_1$ for background+signal . Assuming the Wilk's theorem is applying, the LH ratio test should follow a $\chi^2$ distribution with dof = nb of parameters. The median value of the LH ratio test distribution should tell us if COSI is sensitive or not to this flux value.

For our example we have only one parameter we test : the signal strength . So for a 3 $\sigma$ sensitivity we want to have the median superior or equal to 9.

Keep in mind we are testing only 3 months of observations but we will scale the flux later to 2 years of observation if it is already sensitive after 3 months.

In [32]:

# We start with 100 but higher would be better
# To adjust depending on how fast the LH fit is
# You can also run a script in parallel and save the likelihood ratio test

nbiter = 2 # put this number to 100

LHratiotest = []

for i in range(nbiter) :

    print(f"ITERATION {i}")
    
    #create a signal dataset with the source injector
    spectrum.F.value = F.value
    signal = injector.inject_model(model = model, make_spectrum_plot = False ,data_save_path = None)#,fluctuate=True)

    
    #create a bkg dataset from the bkg model
    bkg_poissonsamp =  Histogram(bg_model.axes)
    bkg_poissonsamp[:] += np.random.poisson(lam = bg_model.to_dense()[:])

    # Set background parameter, which is used to fit the amplitude of the background, and instantiate the COSI 3ML plugin
    # here we define only one bkg model but we could add more
    bkg_dist = {"Background":bg_model.project('Em', 'Phi', 'PsiChi')}
               
    # Workaround to avoid inf values. Out bkg should be smooth, but currently it's not.
    for bckfile in bkg_dist.keys() :
        bkg_dist[bckfile] += sys.float_info.min

    #combine the data + the bck like we would get for real data
    data = EmCDSBinnedData(signal.project('Em', 'Phi', 'PsiChi') 
                           + bkg_poissonsamp.project('Em', 'Phi', 'PsiChi')
                          )
    bkg = FreeNormBinnedBackground(bkg_dist,
                                   sc_history=sc_orientation,
                                   copy = False)



    esr = BinnedThreeMLExtendedSourceResponse(data = data,
                                          precomputed_psr = dr,
                                           
                                           polarization_axis = dr.axes['Pol'] if 'Pol' in dr.axes.labels else None,
                                           )

    ##====


    response = BinnedThreeMLModelFolding(data = data, extended_source_response = esr)

    like_fun = PoissonLikelihood(data, response, bkg)

    cosi = ThreeMLPluginInterface('cosi',
                                  like_fun,
                                  response,
                                  bkg)

    # Nuisance parameter guess, bounds, etc.
    cosi.bkg_parameter['Background'] = Parameter("Background",  # background parameter
                                      1,  # initial value of parameter
                                      min_value=0,  # minimum value of parameter
                                      max_value=10,  # maximum value of parameter
                                      delta=0.05,  # initial step used by fitting engine
                                      unit = u.Hz
                                      )
   
    # Set the source Model
    cosi.set_model(model)


    plugins = DataList(cosi) # If we had multiple instruments, we would do e.g. DataList(cosi, lat, hawc, ...)

    like = JointLikelihood(model, plugins, verbose = False)

    #do the fit
    like.fit()

    #get L1
    L1 = cosi.get_log_like()

    #Get L0
    spectrum.F.value = 0 #null hypothesis (no signal)
    L0 = cosi.get_log_like()
    
    #save LH ratio test
    LHratiotest.append( -2 * (L0-L1) )



ITERATION 0


21:05:42 INFO      set the minimizer to minuit                                             ]8;id=806895;file:///uni-mainz.de/homes/sgallego/software/COSItools/cosipy_interfaces/Pyenv/lib/python3.11/site-packages/threeML/classicMLE/joint_likelihood.py\joint_likelihood.py]8;;\:]8;id=712369;file:///uni-mainz.de/homes/sgallego/software/COSItools/cosipy_interfaces/Pyenv/lib/python3.11/site-packages/threeML/classicMLE/joint_likelihood.py#1017\1017]8;;\

... Reading Extended source response ...
... Reading Extended source response ...
--> done (source name : source)
--> done (source name : source)


Best fit values:

,result,unit
parameter,,
source.spectrum.main.Gaussian.F,(3.4 +/- 0.9) x 10^-5,1 / (s cm2)
Background,(1.204 +/- 0.006) x 10^-2,Hz


...

1.00,-0.75
-0.75,1.00


Values of -log(likelihood) at the minimum:

,-log(likelihood)
cosi,115561.34154906352
total,115561.34154906352


Values of statistical measures:

,statistical measures
AIC,231126.6831632323
BIC,231146.93195544003


## Plot the LH ratio test




In [35]:
if np.median(LHratiotest) <= 9 :
    print(f"COSI is not sensitive to this model with a flux of {F.value} ph/cm2/s. Please try a lower flux")
else :
    print(f"COSI is 3 sigma sensitive to {F.value} ph/cm2/s. You can continue to the next cell")

fig,ax = plt.subplots()
ax.hist(LHratiotest,bins=30)
ax.axvline(np.median(LHratiotest),label=f"median {np.median(LHratiotest):.3f}",color="r",linestyle="--")
ax.legend()
ax.set_xlabel("LH ratio test")
ax.set_ylabel("counts")
ax.set_title(f"Model - flux {F.value} ph/cm2/s")

COSI is 3 sigma sensitive to 3e-05 ph/cm2/s. You can continue to the next cell


Text(0.5, 1.0, 'Model - flux 3e-05 ph/cm2/s')

## Extrapolate the sensitivity to 2 years

DC4 simulations only includes 3 months of orbit. To get the sensitivity after 2 years (nominal life time of COSI) as a first approximation we can scale the flux by $\frac{1}{\sqrt{T}}$    

In [36]:
# 3 * 8 = 24 months = 2 years 

sensi_2y = F.value/np.sqrt(8)

print(f"3 sigma sensitivity after 2 years of COSI observation : {sensi_2y}")


3 sigma sensitivity after 2 years of COSI observation : 1.0606601717798212e-05
